# Project interactive visualization
Using a dataset that your group is consider using for the term project, let's build some meaningful user-driven data visualization. Depending on your dataset this could include:
- Usage of geospatial information
- Building interactive views with widgets
- Organize multiple components into a simple dashboard

Keep these design principles in mind:
While you navigate through this notebook some things to take into consideration:
* Do not add interaction just to add it. Make sure it helps answer a question.
* Use meaningful titles and labels
* Document your code so it's readable and clean. If something does not work, document the issue and explain your best attempt.

In [ ]:
## These are most likely the libraries you will use
# Add or remove imports as needed

import pandas as pd
import numpy as np

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go

# Geospatial / geocoding
from geopy.geocoders import Nominatim

# Panel
import panel as pn
import panel.widgets as pnw
pn.extension('plotly', "tabulator")

!pip install jupyter_bokeh

In [ ]:
### Load your dataset here

# Example:
# df = pd.read_csv("your_dataset.csv")

In [ ]:
# Write your code here
from google.colab import drive
drive.mount('/content/drive')
games_data = "/content/drive/MyDrive/CS133/Video_Games_Sales_as_at_22_Dec_2016.csv"
games = pd.read_csv(games_data,
                      na_values=['tbd'],
                      engine='python',
                     )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
games.head()

,Name,Platform,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating
0,Wii Sports,Wii,2006.0,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8.0,322.0,Nintendo,E
1,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,NaN,NaN,NaN,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,Nintendo,E
3,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8.0,192.0,Nintendo,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,NaN,NaN,NaN,NaN,NaN,NaN


## Question 1: Analytical Question
Write a question about your data that can be explored with an interactive plot. A good example would be a question where you can compare different categories. Explain how the plot help answer your question and why you choose that plot type.

Examples:
- Which regions have the highest average values?
- How does one variable compare across time?

## Answer

Question: How does the distribution of global video game sales differ across platforms?

This question compares the categories of global sales and platforms. We'll be using a box plot because it will help us easily compare different gaming platforms by giving us the median, spread, and outliers for each of them. This is important because video game platforms usually have a few extreme blockbuster games, as well as some games that have significantly lower sales than average. This helps us answer the question since we can compare the medians and spreads of each platform to see how each platform's global sales distributions differ.


## Question 2. Create a simple interaction plot
Create a plot, it can be related to your question #1 or a new question, that users can interact with in some meaningful way. Explain in a markdown, how does the interaction add to the analysis.

Example of possible interactions:
*   Hover over information
*   Toogle between groups

In [ ]:
# Your code . . .
platform_count = games["Platform"].value_counts()
top_platforms = platform_count.head(10).index.tolist()
q1 = games[games["Platform"].isin(top_platforms)].copy()
q1["Log_Sales"] = np.log1p(q1["Global_Sales"])

plot = px.box(
    q1,
    x="Platform",
    y="Log_Sales",
    points="outliers",
    hover_name="Name",
    hover_data={
        "Genre": True,
        "Year_of_Release": True,
        "Global_Sales": ":.2f",
        "Platform": False
    },
    title="Distribution of Global Video Game Sales by Platform (Log Scale)",
    labels={"Log_Sales": "log(1 + Global Sales)", "Platform": "Platform"}
)

plot.show()

## Answer
This plot shows how global video game sales are distributed across different platforms. Users can hover over outlier points to see details about the specific game, including its name, genre, release year, and sales. This interaction adds to the analysis because it helps identify which games are responsible for the extremely high sales values. This lets us understand what the outliers of the plot actually are and how they affect the distribution. We're also able to see the exact numeric quartiles, medians, and minimums of each platform.

## Question 3. Choropleth Planning
Design a choropleth idea using your dataset.
In a markdown cell:
*  Identify the geographic unit you would map (state, county, country, ZIP code, etc.)
*  Identify the variable you would color by
*  Explain if any aggregation or preprocessing is needed
*  Briefly describe what GeoJSON file would be required

You do not need to have a perfect dataset for this question. However, your plan should be realistic.

If your data does not fully support a choropleth, build a prototype table that explains that structure you would need before mapping.

## Answer
A choropleth idea that could work with our dataset would be to map games by the region where they were made.

Geographic unit to map: Region

Variable to color by: Average Sales

Aggregation: The dataset is split into NA, EU, JP, and other sales

GeoJSON file: A GeoJSON file containing the boundary shapes for the regions being mapped, such as North America, Europe, Japan, and Other, would be required.

## Question 4. Geospatial Possibility Check

Determine whether your dataset can support a map-based visualization.

In a markdown cell, answer one of the following:
- If **yes**, explain what geographic fields you have and what type of map is appropriate.
- If **no**, explain what is missing and what you would need to create a map.

Write code that inspects or prepares the geographic column(s) you may use.

It is a bit difficult to create a map with this dataset as the geographic information is inconsistent and sparse. Sales are grouped inconsistently between NA - A group of countries presumably the US primairly, EU - A large group of countries found on that continent, JP - Which is a single country Japan, and other-sales which doesn't particularly specify or clarify what is included in that set. For example about about countries like China?
Some potential geographically identifying data would include things like price per country, publisher region such as japan or america. Anything that could serve to ground these games to a geographic region concretely.

In [ ]:
publisher_sales = games.groupby("Publisher")[[ "NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]].mean()

publisher_sales["Total_Average_Sales"] = publisher_sales["NA_Sales"] + publisher_sales["EU_Sales"] + publisher_sales["JP_Sales"] + publisher_sales["Other_Sales"]

publisher_sales["NA_Sales_Percent"] = (publisher_sales["NA_Sales"] / publisher_sales["Total_Average_Sales"]) * 100
publisher_sales["EU_Sales_Percent"] = (publisher_sales["EU_Sales"] / publisher_sales["Total_Average_Sales"]) * 100
publisher_sales["JP_Sales_Percent"] = (publisher_sales["JP_Sales"] / publisher_sales["Total_Average_Sales"]) * 100
publisher_sales["Other_Sales_Percent"] = (publisher_sales["Other_Sales"] / publisher_sales["Total_Average_Sales"]) * 100

print(publisher_sales[["NA_Sales_Percent", "EU_Sales_Percent", "JP_Sales_Percent", "Other_Sales_Percent"]].head())


                              NA_Sales_Percent  EU_Sales_Percent  \
Publisher                                                          
10TACLE Studios                      63.636364         36.363636   
1C Company                           11.111111         77.777778   
20th Century Fox Video Games         94.300518          5.181347   
2D Boy                                0.000000         75.000000   
3DO                                  63.842365         29.950739   

                              JP_Sales_Percent  Other_Sales_Percent  
Publisher                                                            
10TACLE Studios                            0.0             0.000000  
1C Company                                 0.0            11.111111  
20th Century Fox Video Games               0.0             0.518135  
2D Boy                                     0.0            25.000000  
3DO                                        0.0             6.206897  


## Question 5. Geopy / Location Preparation

If your dataset has location names, addresses, cities, states, or countries, use geopy on a sample of your data to geocode locations or validate location information.

If your dataset does not have data that contains location, pick 5 destination you want to visit and use geopy to geocode locations.

In [ ]:
# Your code . . .
#Our data does not contain location, I'd like to visit Rome, Japan, Iceland, Australia, Austria
destination_names = ["Rome", "Japan", "Iceland", "Australia", "Austria"]
ope = ""
geolocator = Nominatim(user_agent="CS133")
try:
  for destination in destination_names:
    location = geolocator.geocode(destination)
    print(f"{destination}: {location.latitude}, {location.longitude}")
except:
  print("Error")


Rome: 41.8933203, 12.4829321
Japan: 36.5748441, 139.2394179
Iceland: 64.9841821, -18.1059013
Australia: -24.7761086, 134.755
Austria: 47.59397, 14.12456


## Question 6. Panel Widget

Create a Panel Widget that controls something in your analysis such as the ability to choose a column, category, year, etc.

The widget should affect an output such as a plot, table, or summary statistic.

In [ ]:
# Your code . . .
platform_options = sorted(games["Platform"].dropna().unique().tolist())
platform_select = pnw.Select(
    name="Choose a Platform",
    options=platform_options,
    value=platform_options[0]
)

def platform_summary(platform):
  temp = games[games["Platform"] == platform].copy()
  summary = temp.groupby("Genre")["Global_Sales"].agg(["count", "mean", "median", "max"]).reset_index()
  summary.columns = ["Genre", "Game Count", "Mean Sales", "Median Sales", "Max Sales"]
  return summary.sort_values("Mean Sales", ascending=False).reset_index(drop=True)

platform_table = pn.bind(platform_summary, platform=platform_select)

pn.Column(
    platform_select,
    pn.widgets.Tabulator(
            platform_table,
            show_index=False,
            width=700,
            height=400,
            theme="default"
        )
)


Column
    [0] Select(name='Choose a Platform', options=['2600', '3DO', ...], value='2600')
    [1] Tabulator(height=400, show_index=False, sizing_mode='fixed', theme='default', value=        Genre  ..., width=700)

## Question 7. Mini Dashboard

Build a small Panel dashboard with at least two components. Arrange the components so that it is in a readable layout. Your dashboard should include:
* At least one plot,
* An additional element of your choice such as a widget, table, second plot, etc.

In [ ]:
# Write your code here
platform_options = sorted(games["Platform"].dropna().unique().tolist())

platform_multi = pnw.MultiChoice(
    name="Platforms",
    value=[platform_options[0]],
    options=platform_options
)

year_slider = pnw.IntRangeSlider(
    name="Release Year Range",
    start=int(games["Year_of_Release"].min(skipna=True)),
    end=int(games["Year_of_Release"].max(skipna=True)),
    value=(
        int(games["Year_of_Release"].min(skipna=True)),
        int(games["Year_of_Release"].max(skipna=True)),
    )
)

def dashboard_plot(platforms, year_range):
    temp = games.copy()
    temp = temp[temp["Platform"].isin(platforms)]
    temp = temp[
        (temp["Year_of_Release"] >= year_range[0]) &
        (temp["Year_of_Release"] <= year_range[1])
    ]

    temp["Log_Sales"] = np.log1p(temp["Global_Sales"])

    plot = px.box(
        temp,
        x="Platform",
        y="Log_Sales",
        points="outliers",
        hover_name="Name",
        hover_data={
            "Genre": True,
            "Year_of_Release": True,
            "Global_Sales": ":.2f",
            "Platform": False
        },
        title="Distribution of Global Video Game Sales by Platform (Log Scale)",
        labels={"Log_Sales": "log(1 + Global Sales)", "Platform": "Platform"}
    )

    return plot

def dashboard_table(platforms, year_range):
  temp = games.copy()
  temp = temp[temp["Platform"].isin(platforms)]
  temp = temp[
      (temp["Year_of_Release"] >= year_range[0]) & (temp["Year_of_Release"] <= year_range[1])
  ]

  return temp[["Name", "Platform", "Genre", "Year_of_Release", "Global_Sales"]].head(15)

dashboardplot = pn.bind(dashboard_plot, platforms=platform_multi, year_range=year_slider)
dashboardtable = pn.bind(dashboard_table, platforms=platform_multi, year_range=year_slider)

dashboard = pn.Column(
    "## Mini Dashboard",
    pn.Row(platform_multi, year_slider),
    pn.Row(
        pn.pane.Plotly(dashboardplot, config={"responsive": True}),
        pn.widgets.Tabulator(
            dashboardtable,
            show_index=False,
            width=700,
            height=400,
            theme="default"
        )
    )
)

dashboard

Column
    [0] Markdown(str)
    [1] Row
        [0] MultiChoice(name='Platforms', options=['2600', '3DO', ...], value=['2600'])
        [1] IntRangeSlider(end=2020, name='Release Year Range', start=1980, value=(1980, 2020), value_end=2020, value_start=1980)
    [2] Row
        [0] Plotly(Figure, config={'responsive': True})
        [1] Tabulator(height=400, show_index=False, sizing_mode='fixed', theme='default', value=              ..., width=700)

## Question 8. Reflection

Write a short reflection addressing all of the following:
- Which interactive element was most useful in your notebook?
- What was the hardest part of working with your dataset?
- Did implementing interactive tools help your analysis? Why or why not?
- If you had more time, what would you improve or add?

## Answer
The most useful interactive element in this notebook was the combination of Panel widgets and Plotly hover interactions.
The widgets made it easy to filter the data by platform and year, and the hover tools let us inspect specific games.

The hardest part of working with this dataset was that it does not include detailed locations.
The database only has regional sales, but no exact country or city data.

Implementing interactive tools did help the analysis because it allowed us to explore patterns dynamically rather than just a single static chart.
For example, we were able to see how sales distributions change over time by different time ranges.

If we had more time we would add more filters, such as for genre and critic score.
We would also look for another dataset with country-level sales to build a choropleth map.